# 01 — Read and join the Olist tables

**Job:** inspect every database table, verify its grain and keys, then create one ML row
per order. Tables with many rows per order (`order_items`, `payments`) are aggregated
before joining so they cannot duplicate orders.

**Reads:** local MySQL database.  
**Writes:** `artifacts/01_join/ml_orders.csv` and `table_profile.csv`.

In [56]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from config import get_engine

OUT = ROOT / "artifacts" / "01_join"
OUT.mkdir(parents=True, exist_ok=True)
engine = get_engine()
pd.set_option("display.max_columns", 100)

In [18]:
tables = pd.read_sql("SHOW TABLES", engine).iloc[:, 0].tolist()
expected_tables = {
    "olist_customers_dataset", "olist_geolocation_dataset",
    "olist_order_items_dataset", "olist_order_payments_dataset",
    "olist_order_reviews_dataset", "olist_orders_dataset",
    "olist_products_dataset", "olist_sellers_dataset",
    "product_category_name_translation",
}
assert expected_tables.issubset(tables), expected_tables - set(tables)

key_candidates = {
    "olist_customers_dataset": ["customer_id"],
    "olist_geolocation_dataset": ["geolocation_zip_code_prefix"],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id"],
    "olist_orders_dataset": ["order_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "product_category_name_translation": ["product_category_name"],
}
profiles = []
for table in sorted(expected_tables):
    count = int(pd.read_sql(f"SELECT COUNT(*) AS n FROM `{table}`", engine).iloc[0, 0])
    sample = pd.read_sql(f"SELECT * FROM `{table}` LIMIT 5", engine)
    keys = key_candidates[table]
    duplicate_keys = int(pd.read_sql(
        f"SELECT COUNT(*) AS n FROM (SELECT {', '.join(f'`{k}`' for k in keys)}, "
        f"COUNT(*) c FROM `{table}` GROUP BY {', '.join(f'`{k}`' for k in keys)} "
        "HAVING c > 1) duplicates", engine
    ).iloc[0, 0])
    profiles.append({
        "table": table, "rows": count, "columns": len(sample.columns),
        "candidate_key": ", ".join(keys), "duplicate_key_groups": duplicate_keys,
        "one_row_means": {
            "olist_customers_dataset": "one order-specific customer record",
            "olist_geolocation_dataset": "one geocoding observation for a ZIP prefix",
            "olist_order_items_dataset": "one item position within an order",
            "olist_order_payments_dataset": "one payment attempt/method within an order",
            "olist_order_reviews_dataset": "one submitted review",
            "olist_orders_dataset": "one order",
            "olist_products_dataset": "one product",
            "olist_sellers_dataset": "one seller",
            "product_category_name_translation": "one category translation",
        }[table],
    })
    display(sample.head(2))

table_profile = pd.DataFrame(profiles)
table_profile.to_csv(OUT / "table_profile.csv", index=False)
display(table_profile)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories


,table,rows,columns,candidate_key,duplicate_key_groups,one_row_means
0,olist_customers_dataset,99441,5,customer_id,0,one order-specific customer record
1,olist_geolocation_dataset,1000163,5,geolocation_zip_code_prefix,17972,one geocoding observation for a ZIP prefix
2,olist_order_items_dataset,112650,7,"order_id, order_item_id",0,one item position within an order
3,olist_order_payments_dataset,103886,5,"order_id, payment_sequential",0,one payment attempt/method within an order
4,olist_order_reviews_dataset,99224,7,review_id,789,one submitted review
5,olist_orders_dataset,99441,8,order_id,0,one order
6,olist_products_dataset,32951,9,product_id,0,one product
7,olist_sellers_dataset,3095,4,seller_id,0,one seller
8,product_category_name_translation,71,2,product_category_name,0,one category translation


In [54]:
orders = pd.read_sql("SELECT * FROM olist_orders_dataset", engine)
customers = pd.read_sql("SELECT * FROM olist_customers_dataset", engine)
items = pd.read_sql("SELECT * FROM olist_order_items_dataset", engine)
payments = pd.read_sql("SELECT * FROM olist_order_payments_dataset", engine)
products = pd.read_sql("SELECT * FROM olist_products_dataset", engine)
sellers = pd.read_sql("SELECT * FROM olist_sellers_dataset", engine)
translations = pd.read_sql("SELECT * FROM product_category_name_translation", engine)
geo = pd.read_sql('''
    SELECT geolocation_zip_code_prefix,
           AVG(geolocation_lat) AS latitude,
           AVG(geolocation_lng) AS longitude
    FROM olist_geolocation_dataset
    GROUP BY geolocation_zip_code_prefix
''', engine)

products = products.merge(translations, on="product_category_name", how="left")
products["product_volume_cm3"] = (
    products["product_length_cm"] * products["product_height_cm"] * products["product_width_cm"]
)
seller_geo = sellers.merge(
    geo.add_prefix("seller_").rename(columns={"seller_geolocation_zip_code_prefix": "seller_zip_code_prefix"}),
    on="seller_zip_code_prefix", how="left",
)
enriched_items = (items
    .merge(products, on="product_id", how="left", validate="many_to_one")
    .merge(seller_geo, on="seller_id", how="left", validate="many_to_one"))

item_agg = enriched_items.groupby("order_id", as_index=False).agg(
    item_count=("order_item_id", "count"),
    unique_product_count=("product_id", "nunique"),
    unique_seller_count=("seller_id", "nunique"),
    total_item_value=("price", "sum"),
    mean_item_price=("price", "mean"),
    total_freight_value=("freight_value", "sum"),
    mean_product_weight_g=("product_weight_g", "mean"),
    mean_product_volume_cm3=("product_volume_cm3", "mean"),
    mean_product_photos_qty=("product_photos_qty", "mean"),
    mean_seller_latitude=("seller_latitude", "mean"),
    mean_seller_longitude=("seller_longitude", "mean"),
)
representative_item = (enriched_items.sort_values(["order_id", "price"], ascending=[True, False])
    .drop_duplicates("order_id")[["order_id", "seller_state", "product_category_name_english"]]
    .rename(columns={"seller_state": "primary_seller_state",
                     "product_category_name_english": "primary_product_category"}))
item_agg = item_agg.merge(representative_item, on="order_id", how="left", validate="one_to_one")

payment_agg = payments.groupby("order_id", as_index=False).agg(
    payment_count=("payment_sequential", "count"),
    payment_type_count=("payment_type", "nunique"),
    max_payment_installments=("payment_installments", "max"),
    total_payment_value=("payment_value", "sum"),
)
primary_payment = (payments.sort_values(["order_id", "payment_value"], ascending=[True, False])
    .drop_duplicates("order_id")[["order_id", "payment_type"]]
    .rename(columns={"payment_type": "primary_payment_type"}))
payment_agg = payment_agg.merge(primary_payment, on="order_id", how="left", validate="one_to_one")

In [57]:
customer_geo = geo.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "latitude": "customer_latitude", "longitude": "customer_longitude",
})
customer_features = customers.merge(
    customer_geo, on="customer_zip_code_prefix", how="left", validate="many_to_one"
)
ml_orders = (orders
    .merge(customer_features, on="customer_id", how="left", validate="many_to_one")
    .merge(item_agg, on="order_id", how="left", validate="one_to_one")
    .merge(payment_agg, on="order_id", how="left", validate="one_to_one"))

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

ml_orders["seller_customer_distance_km"] = haversine_km(
    ml_orders["customer_latitude"], ml_orders["customer_longitude"],
    ml_orders["mean_seller_latitude"], ml_orders["mean_seller_longitude"],
)
assert ml_orders["order_id"].is_unique
assert len(ml_orders) == len(orders)
ml_orders.to_csv(OUT / "ml_orders.csv", index=False)
print(f"Saved {len(ml_orders):,} unique orders with {ml_orders.shape[1]} columns")
display(ml_orders.head())

Saved 99,441 unique orders with 33 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_latitude,customer_longitude,item_count,unique_product_count,unique_seller_count,total_item_value,mean_item_price,total_freight_value,mean_product_weight_g,mean_product_volume_cm3,mean_product_photos_qty,mean_seller_latitude,mean_seller_longitude,primary_seller_state,primary_product_category,payment_count,payment_type_count,max_payment_installments,total_payment_value,primary_payment_type,seller_customer_distance_km
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,-23.576983,-46.587161,1.0,1.0,1.0,29.99,29.99,8.72,500.0,1976.0,4.0,-23.680729,-46.444238,SP,housewares,3.0,2.0,1.0,38.71,voucher,18.576110
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,-12.177924,-44.660711,1.0,1.0,1.0,118.70,118.70,22.76,400.0,4693.0,1.0,-19.807681,-43.980427,SP,perfumery,1.0,1.0,1.0,141.46,boleto,851.495069
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,-16.745150,-48.514783,1.0,1.0,1.0,159.90,159.90,19.22,420.0,9576.0,1.0,-21.363502,-48.229601,SP,auto,1.0,1.0,3.0,179.12,credit_card,514.410666
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,-5.774190,-35.271143,1.0,1.0,1.0,45.00,45.00,27.20,450.0,6000.0,3.0,-19.837682,-43.924053,MG,pet_shop,1.0,1.0,1.0,72.20,credit_card,1822.226336
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,-23.676370,-46.514627,1.0,1.0,1.0,19.90,19.90,8.72,250.0,11475.0,4.0,-23.543395,-46.262086,SP,stationery,1.0,1.0,1.0,28.62,credit_card,29.676625
